In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.iv import compare
from linearmodels.panel.results import compare
import statsmodels.api as sm
from statsmodels.api import qqplot
from linearmodels.panel import PanelOLS
from scipy import stats as scipy_stats

import pyreadstat as prs
import pygwalker as pyg

import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

import json

from tqdm.notebook import tqdm
from itables import init_notebook_mode
init_notebook_mode()

# working pop preparations

In [ ]:
additional_health_variables = pl.read_excel('../data/disease_variables.xlsx')
additional_health_variables

In [ ]:
cols_disease = additional_health_variables['var'].to_list()
cols_disease

In [ ]:
cols = ['idind', 'year', 'h7_2', 'h7_1', 'origsm', 'region', 'psu', 'status', 'age', 'educ', 'marst', 'h5', 'm20_7', 'j1', 'j60', 'j40', 'j363']

In [ ]:
cols + cols_disease

In [ ]:
panel = pl.read_parquet('../data/RLMS_IND_2015_2024_eng_prepared.parquet', columns=cols + cols_disease)
for var, label, fill in zip(additional_health_variables['var'], additional_health_variables['label'], additional_health_variables['fill']):
    panel = panel.with_columns(
        pl.col(var).fill_null(fill).alias(label)
    )
panel

In [ ]:
panel = panel.with_columns(
    date = pl.concat_str([pl.col('year').cast(pl.Int32), pl.col('h7_2'), pl.col('h7_1').cast(pl.Int32)], separator=' ').str.strptime(pl.Date, format='%Y %B %d')
)
panel

In [ ]:
panel = panel.with_columns(
    is_employed = pl.when(pl.col('j1').is_in(['You are currently working', 'You are on paid leave: maternity leave or taking care of a child under 3 years o', 'You are on another kind of paid leave'])).then(1).otherwise(0)
)
panel

In [ ]:
panel = panel.with_columns(
    has_disablity = pl.when(pl.col('m20_7').is_in(['No', 'NO ANSWER'])).then(0).otherwise(1)
)
panel

In [ ]:
panel = panel.with_columns(
    is_employed_has_disablity = pl.col('is_employed') * pl.col('has_disablity')
)
panel

In [ ]:
panel = panel.with_columns(
    is_employed_no_disablity = pl.col('is_employed') * pl.col('has_disablity').replace({0:1, 1:0})
)
panel

In [ ]:
panel = panel.with_columns(
    net_job = pl.col('j60').fill_null(0) - pl.col('j363').fill_null(0)
)
panel = panel.with_columns(
    pl.when(pl.col('net_job') < 0).then(0).otherwise('net_job').alias('net_job')
)
panel

In [ ]:
panel  = panel.filter(pl.col('net_job') < 1_000_000)
panel

In [ ]:
(
    panel
        .filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

In [ ]:
panel

In [ ]:

len(panel.unique('idind'))

In [ ]:
panel = panel.with_columns(
    working_pop = pl.when((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1)).then(1).otherwise(0)
)
panel

In [ ]:
panel_working_pop_2019 = panel.filter(
    #pl.col('date').is_between(pl.date(2019, 1, 1), pl.date(2019, 12, 31))
    pl.col('year') == 2019
)
panel_working_pop_2019

In [ ]:
panel_working_pop_2020 = panel.filter(
    #pl.col('date').is_between(pl.date(2020, 1, 1), pl.date(2020, 12, 31))
    pl.col('year') == 2020
)
panel_working_pop_2020

In [ ]:
panel_working_pop_2020['status'].value_counts()

In [ ]:
common_people = panel_working_pop_2019[['idind']].join(panel_working_pop_2020[['idind']], how='inner', on='idind').unique()
common_people

In [ ]:
len(common_people)

In [ ]:
len(common_people.unique())

In [ ]:
panel_common_2019_2020 = panel.filter(
    pl.col('idind').is_in(common_people['idind'].implode())
)
panel_common_2019_2020

In [ ]:
panel_common_2019_2020['h7_2'].value_counts()

In [ ]:
len(panel_common_2019_2020)

In [ ]:
# panel_common_2019_2020.write_parquet('../data/panel_common_2019_2020.parquet')

In [ ]:
panel_common_2019_2020.filter(
    pl.col('is_employed_no_disablity') == 1
).group_by('year').len().plot.line(x='year', y='len')

In [ ]:
panel_common_2019_2020.filter(
    pl.col('is_employed_has_disablity') == 1
).group_by('year').len().plot.line(x='year', y='len')

In [ ]:
(
    panel_common_2019_2020
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        .filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
        .filter(pl.col('working_pop') == 1)
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

In [ ]:
alt.data_transformers.enable("vegafusion")
panel_common_2019_2020.filter(pl.col('h7_2').is_in(['October', 'November', 'December'])).plot.line(x='date', y='net_job', color='has_disablity')

In [ ]:
(
    panel_common_2019_2020
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        #.filter(pl.col('working_pop') == 1)
        .filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

In [ ]:
len(panel_common_2019_2020.unique('idind'))

In [ ]:
did_data = panel_common_2019_2020.filter(pl.col('working_pop') == 1).filter(
    (pl.col('year').is_in([2017, 2018 ,2019, 2020]))
)[['idind', 'year', 'net_job', 'is_employed_has_disablity']].to_pandas().set_index(['idind', 'year'])
did_data.sort_index()

In [ ]:
X_agg = did_data[['is_employed_has_disablity']]
y = did_data['net_job']

In [ ]:
agg_did_model = PanelOLS(y, X_agg, entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(agg_did_model.summary)

# psm

In [ ]:
education_levels = pl.read_excel('../data/all_education_levels.xlsx', sheet_name='selected_levels')
education_levels

In [ ]:
marital_status = pl.read_excel('../data/all_marst_levels.xlsx', sheet_name='selected_levels')
marital_status

In [ ]:
period_relevance = pl.read_excel('../data/period_relevance.xlsx').with_columns(
    pl.col('year').cast(pl.Float64)
)
period_relevance

In [ ]:
is_town = pl.read_excel('../data/is_town.xlsx')
is_town

In [ ]:
prepsm_data = (
    panel_common_2019_2020
        .filter(pl.col('working_pop') == 1)
        .join(education_levels, on='educ', how='left', validate='m:1')
        .with_columns(pl.col('educ_level').fill_null('school_or_less'))
        .join(marital_status, on='marst', how='left', validate='m:1')
        .join(period_relevance, on='year', how='left', validate='m:1')
        .join(is_town, on='status', how='left', validate='m:1')
        .with_columns(pl.col('is_married').fill_null(0))
        .with_columns(
            is_female = pl.when(pl.col('h5') == 'FEMALE').then(1).otherwise(0),
        ).with_columns(
            pl.when(pl.col('chronic_heart') == 'Yes').then(1).otherwise(0).alias('chronic_heart'),
            pl.when(pl.col('chronic_lung') == 'Yes').then(1).otherwise(0).alias('chronic_lung'),
            pl.when(pl.col('chronic_liver') == 'Yes').then(1).otherwise(0).alias('chronic_liver'),
            pl.when(pl.col('chronic_kidney') == 'Yes').then(1).otherwise(0).alias('chronic_kidney'),
            pl.when(pl.col('chronic_stomach') == 'Yes').then(1).otherwise(0).alias('chronic_stomach'),
            pl.when(pl.col('chronic_spinal') == 'Yes').then(1).otherwise(0).alias('chronic_spinal'),
            pl.when(pl.col('disease_neuro') == 'Yes').then(1).otherwise(0).alias('disease_neuro'),
            pl.when(pl.col('disease_eye') == 'Yes').then(1).otherwise(0).alias('disease_eye'),
            pl.when(pl.col('allegies') == 'Yes').then(1).otherwise(0).alias('allegies')
        )
).drop(cols_disease)
prepsm_data

In [ ]:
prepsm_data['status'].value_counts()

In [ ]:
prepsm_data['is_town'].value_counts()

In [ ]:
prepsm_data['educ_level'].value_counts()

In [ ]:
# prepsm_data['marst'].value_counts().write_excel('../data/all_marst_levels.xlsx')
prepsm_data = (
    prepsm_data
        .rename({'status':'habitat'}).to_dummies('habitat')
        .to_dummies('educ_level')
        .to_dummies('disability_class')
        .drop('disability_class_no')
        .rename({'disability_class_First group':'disability_class_first_group', 'disability_class_Second group':'disability_class_second_group', 'disability_class_Third group':'disability_class_third_group'})
)
prepsm_data

In [ ]:
prepsm_data = (
    prepsm_data
        .filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
    .group_by(['idind', 'year']).agg(
        pl.col('j60').mean(), pl.col('net_job').mean(), pl.col('is_employed_has_disablity').last(), pl.col('age').last(), pl.col('is_female').last(), pl.col('is_town').last(), pl.col('educ_level_school_or_less').last(), pl.col('educ_level_common').last(), pl.col('educ_level_higher').last(), pl.col('educ_level_university').last(), pl.col('is_married').last(), pl.col('chronic_heart').last(), pl.col('chronic_lung').last(), pl.col('chronic_liver').last(), pl.col('chronic_kidney').last(), pl.col('chronic_stomach').last(), pl.col('chronic_spinal').last(), pl.col('disease_neuro').last(), pl.col('disease_eye').last(), pl.col('allegies').last(), pl.col('disability_class_first_group').last(), pl.col('disability_class_second_group').last(), pl.col('disability_class_third_group').last(), pl.col('period_relevance').last()
    )
    .sort(['idind', 'year'])
)
prepsm_data

In [ ]:
prepsm_data.filter(pl.col('is_employed_has_disablity') == 1)['net_job'].describe()

In [ ]:
prepsm_data.filter(pl.col('is_employed_has_disablity') == 1).plot.boxplot(y='net_job').properties(width=100, height=350)

In [ ]:
prepsm_data.filter(pl.col('is_employed_has_disablity') == 0)['net_job'].describe()

In [ ]:
prepsm_data.filter(pl.col('is_employed_has_disablity') == 0).plot.boxplot(y='net_job').properties(width=100, height=350)

In [ ]:
ids_to_exclude_0 = prepsm_data.filter(pl.col('is_employed_has_disablity') == 0).filter(pl.col('net_job') > 500_000)['idind'].to_list()
ids_to_exclude_0

In [ ]:
ids_to_exclude_1 = prepsm_data.filter(pl.col('is_employed_has_disablity') == 1).filter(pl.col('net_job') > 120000)['idind'].to_list()
ids_to_exclude_1 = [x for x in ids_to_exclude_1 if x not in [7399, 40749]]
ids_to_exclude_1

In [ ]:
(
    prepsm_data
        #.filter(~pl.col('idind').is_in(possible_spillovers.implode()))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('idind').len())
).plot.line(x='year', y='idind', color='is_employed_has_disablity')

In [ ]:
(
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
(
    prepsm_data
        #.filter(~pl.col('idind').is_in(possible_spillovers.implode()))
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean().log())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
(
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([2017,  2018, 2019, 2020, 2021]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
prepsm_data.filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1)).filter(pl.col('is_employed_has_disablity') == 1).plot.boxplot(y='net_job').properties(width=100, height=350)

In [ ]:
prepsm_data.filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1)).filter(pl.col('is_employed_has_disablity') == 0).plot.boxplot(y='net_job').properties(width=100, height=350)

In [ ]:
panel2019_2020 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas().set_index(['idind', 'year'])
panel2019_2020

In [ ]:
model1 = PanelOLS.from_formula('np.log(net_job + 1) ~ is_employed_has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model1.summary)

In [ ]:
model2 = PanelOLS.from_formula('net_job ~ is_employed_has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model2.summary)

In [ ]:
model3 = PanelOLS.from_formula('net_job ~ is_employed_has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model3.summary)

In [ ]:
panel2018_2019 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([2018, 2019]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas().set_index(['idind', 'year'])
panel2018_2019

In [ ]:
# model3 = PanelOLS.from_formula('np.log(net_job + 1) ~ is_employed_has_disablity + EntityEffects', data=panel2018_2019).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
# print(model3.summary)

In [ ]:
# model4 = PanelOLS.from_formula('net_job ~ is_employed_has_disablity + EntityEffects', data=panel2018_2019).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
# print(model4.summary)

In [ ]:
print(compare({'19-20 log':model1, "19-20": model2}, stars=True))

In [ ]:
panel_2014_2024 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        # .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas()
panel_2014_2024_dta = panel_2014_2024.set_index(['idind', 'year'])
panel_2014_2024_dta

In [ ]:
def wald_test_for_period_insignificance(event_study_model_reduced):
    pre_treatment_terms = [t for t in event_study_terms if int(t.split('_')[2]) < -1]
    pre_coefs = event_study_model_reduced.params[pre_treatment_terms].values
    pre_vcov = event_study_model_reduced.cov.loc[pre_treatment_terms, pre_treatment_terms].values

    q = len(pre_treatment_terms)
    wald_stat = pre_coefs @ np.linalg.inv(pre_vcov) @ pre_coefs / q
    df_resid = event_study_model_reduced.nobs - event_study_model_reduced.params.shape[0]
    p_value = 1 - scipy_stats.f.cdf(wald_stat, q, df_resid)
    
    print(f"\nJoint F-test for pre-treatment coefficients:")
    print(f"  F({q}, {df_resid}) = {wald_stat:.4f}")
    print(f"  p-value = {p_value:.4f}")

In [ ]:
event_study_terms = []
for p in sorted(panel_2014_2024['period_relevance'].unique()):
    if p != -1:  # Exclude reference period
        col_name = f'rel_period_{p}_treated'
        # Create interaction: (relative_period == p) * treatment
        panel_2014_2024_dta[col_name] = ((panel_2014_2024['period_relevance'].values == p) & (panel_2014_2024['is_employed_has_disablity'].values == 1)).astype(int)
        event_study_terms.append(col_name)

X_event = panel_2014_2024_dta[event_study_terms]
y_event = panel_2014_2024_dta['net_job']

event_study_model = PanelOLS(y_event, X_event, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(event_study_model.summary)

In [ ]:
wald_test_for_period_insignificance(event_study_model)

In [ ]:
event_study_terms = []
for p in sorted(panel_2014_2024['period_relevance'].unique()):
    if p != -1:  # Exclude reference period
        col_name = f'rel_period_{p}_treated'
        # Create interaction: (relative_period == p) * treatment
        panel_2014_2024_dta[col_name] = ((panel_2014_2024['period_relevance'].values == p) & (panel_2014_2024['is_employed_has_disablity'].values == 1)).astype(int)
        event_study_terms.append(col_name)

X_event = panel_2014_2024_dta[event_study_terms]
y_event = np.log(panel_2014_2024_dta['net_job'] + 1)

event_study_model_log = PanelOLS(y_event, X_event, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(event_study_model_log.summary)

wald_test_for_period_insignificance(event_study_model_log)

In [ ]:
alike_matched = pl.read_parquet('../data/panel_common_post_psm_full_mean3_months_with_add_health_variables_no_outliers.parquet').filter(pl.col('is_employed_has_disablity') == 0)['idind']#.to_pandas().set_index(['idind', 'year'])

In [ ]:
panel2019_2020 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas().set_index(['idind', 'year'])
panel2019_2020

In [ ]:
model5 = PanelOLS.from_formula('np.log(net_job + 1) ~ is_employed_has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model5.summary)

In [ ]:
model6 = PanelOLS.from_formula('net_job ~ is_employed_has_disablity + EntityEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model6.summary)

In [ ]:
panel_2014_2024 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(pl.col('period_relevance') > -5)
        # .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas()
panel_2014_2024_dta = panel_2014_2024.set_index(['idind', 'year'])
panel_2014_2024_dta

In [ ]:
event_study_terms = []
for p in sorted(panel_2014_2024['period_relevance'].unique()):
    if p != -1:  # Exclude reference period
        col_name = f'rel_period_{p}_treated'
        # Create interaction: (relative_period == p) * treatment
        panel_2014_2024_dta[col_name] = ((panel_2014_2024['period_relevance'].values == p) & (panel_2014_2024['is_employed_has_disablity'].values == 1)).astype(int)
        event_study_terms.append(col_name)

X_event = panel_2014_2024_dta[event_study_terms]
y_event = panel_2014_2024_dta['net_job']

event_study_model_reduced = PanelOLS(y_event, X_event, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(event_study_model_reduced.summary)

wald_test_for_period_insignificance(event_study_model_reduced)

In [ ]:
event_study_terms = []
for p in sorted(panel_2014_2024['period_relevance'].unique()):
    if p != -1:  # Exclude reference period
        col_name = f'rel_period_{p}_treated'
        # Create interaction: (relative_period == p) * treatment
        panel_2014_2024_dta[col_name] = ((panel_2014_2024['period_relevance'].values == p) & (panel_2014_2024['is_employed_has_disablity'].values == 1)).astype(int)
        event_study_terms.append(col_name)

X_event = panel_2014_2024_dta[event_study_terms]
y_event = np.log(panel_2014_2024_dta['net_job'] + 1)

event_study_model_reduced_log = PanelOLS(y_event, X_event, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(event_study_model_reduced_log.summary)

wald_test_for_period_insignificance(event_study_model_reduced_log)

In [ ]:
print(compare({'19-20 T':model2, "19-20 RA": model6, "19-20 T log": model1, '19-20 RA log':model5}, stars=True))

In [ ]:
(
    prepsm_data
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('year').is_in([ 2018, 2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
print(compare({'19-20 T':event_study_model, "19-20 RA": event_study_model_reduced, "19-20 T log": event_study_model_log, '19-20 RA log':event_study_model_reduced_log}, stars=True))

In [ ]:
(
    prepsm_data
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        #.filter(pl.col('year').is_in([ 2018, 2019, 2020]))
        .filter(pl.col('period_relevance') > -5)
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
(
    prepsm_data
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(pl.col('period_relevance') > -5)
        #.filter(pl.col('year').is_in([ 2018, 2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['period_relevance', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='period_relevance', y='net_job', color='is_employed_has_disablity')

In [ ]:
book_colors = {
    'primary': '#2E86AB',    # Steel blue - main data
    'secondary': '#A23B72',  # Magenta - secondary data
    'accent': '#F18F01',     # Orangxe - highlights/warnings
    'success': '#C73E1D',    # Red-orange - thresholds/targets
    'muted': '#6C757D',      # Gray - reference lines
    'light_gray': '#E5E5E5', # Light gray - backgrounds
    'dark_gray': '#4D4D4D'   # Dark gray - text
}

def setup_book_style():
    """Apply consistent styling to matplotlib plots"""
    plt.rcParams.update({
        'figure.figsize': (10, 7),
        'figure.dpi': 100,
        'font.size': 12,
        'axes.titlesize': 16,
        'axes.titleweight': 'bold',
        'axes.labelsize': 14,
        'axes.labelcolor': '#4D4D4D',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'grid.color': '#E5E5E5',
        'legend.fontsize': 11,
        'legend.frameon': False,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
    })

setup_book_style()

In [ ]:
def plot_event_margins(model, measure_unit = 'RUB'):
    coef_names = [col for col in model.params.index if col.startswith('rel_period')]
    coefs = model.params[coef_names]
    std_errors = model.std_errors[coef_names]
    
    # Parse relative periods from coefficient names
    rel_periods = [int(name.split('_')[2]) for name in coef_names]
    
    # Create DataFrame for plotting
    coef_df = pd.DataFrame({
        'relative_period': rel_periods,
        'estimate': coefs.values,
        'std_error': std_errors.values
    })
    
    # Add reference period (k = -1) with zero effect
    ref_row = pd.DataFrame({
        'relative_period': [-1],
        'estimate': [0],
        'std_error': [0]
    })
    coef_df = pd.concat([coef_df, ref_row], ignore_index=True)
    coef_df = coef_df.sort_values('relative_period').reset_index(drop=True)
    
    # Calculate confidence intervals (95%)
    coef_df['ci_lower'] = coef_df['estimate'] - 1.96 * coef_df['std_error']
    coef_df['ci_upper'] = coef_df['estimate'] + 1.96 * coef_df['std_error']
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Shade pre-treatment region
    ax.axvspan(-6.5, -0.5, alpha=0.15, color=book_colors['light_gray'], zorder=0)
    
    # Reference lines
    ax.axhline(y=0, linestyle='--', color=book_colors['muted'], linewidth=1, alpha=0.8)
    ax.axvline(x=-0.5, linestyle='--', color=book_colors['muted'], linewidth=1, alpha=0.8)

    ax.axvline(x=-1, linestyle='--', color='red', linewidth=1, alpha=0.8)
    ax.axvline(x=1, linestyle='--', color='red', linewidth=1, alpha=0.8)

    # ax.annotate('pandemic restrictions\nperiod of interest',
    #             xy=(0, coef_df['ci_upper'].min() * 0.5),
    #             fontsize=10, color='red', ha='center', style='italic')
    
    # Error bars
    ax.errorbar(
        coef_df['relative_period'],
        coef_df['estimate'],
        yerr=[coef_df['estimate'] - coef_df['ci_lower'],
              coef_df['ci_upper'] - coef_df['estimate']],
        fmt='o',
        color=book_colors['primary'],
        markersize=8,
        capsize=4,
        capthick=1.5,
        linewidth=1.5,
        ecolor=book_colors['primary'],
        zorder=3
    )
    
    # Annotations
    ax.annotate('Pre-pandemic',
                xy=(-3.5, coef_df['ci_upper'].max() * 0.6),
                fontsize=10, color=book_colors['muted'], ha='center', style='italic')
    
    ax.annotate('Post-pandemic',
                xy=(2.5, coef_df['ci_upper'].max() * 0.95),
                fontsize=10, color=book_colors['primary'], ha='center', style='italic')
    
    # Labels and title
    ax.set_xlabel('Periods', fontsize=14, color=book_colors['dark_gray'])
    ax.set_ylabel(f'Estimated effect ({measure_unit})', fontsize=14, color=book_colors['dark_gray'])
    ax.set_title('Effect of covid-19 restrictions on NEI',
                 fontsize=16, fontweight='bold', color='#333333', pad=15)
    
    # Subtitle
    ax.text(0.5, 1.02, 'Reference period: k = -1 (one period before treatment)',
            transform=ax.transAxes, fontsize=11, color='grey', ha='center')
    
    # Caption (takeaway)
    fig.text(0.1, -0.02,
             'Takeaway: No significant pre-trends; effect peaks at k=1, then gradually declines.',
             fontsize=9, color='grey', ha='left')
    
    ax.set_xticks(range(-6, 4))
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.12)
    plt.show()

In [ ]:
plot_event_margins(event_study_model)

In [ ]:
plot_event_margins(event_study_model_reduced, measure_unit='%%')

In [ ]:
plot_event_margins(event_study_model_log)

In [ ]:
plot_event_margins(event_study_model_reduced_log)

# autocorrelation test

In [ ]:
panel_2014_2024 = (
    prepsm_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .filter(~pl.col('idind').is_in(alike_matched.implode()))
        .filter(pl.col('period_relevance') > -5)
        # .filter(pl.col('year').is_in([2019, 2020]))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas()
panel_2014_2024_dta = panel_2014_2024.set_index(['idind', 'year'])
panel_2014_2024_dta

In [ ]:
fe_model = PanelOLS(
    panel_2014_2024_dta['net_job'],
    sm.add_constant(panel_2014_2024_dta[['period_relevance']].astype(float)),
    entity_effects=True
)

fe_resid = fe_model.fit().resids

In [ ]:
fe_model.fit().resids.to_frame().boxplot(return_type='axes')

In [ ]:
fe_model.fit().resids.describe()

In [ ]:
resid_df = fe_model.fit().resids.to_frame()
resid_df

In [ ]:
resid_df['resid_diff'] = resid_df.groupby('idind')['residual'].diff()
resid_df['resid_diff_lag'] = resid_df.groupby('idind')['resid_diff'].shift(1)
resid_df = resid_df.dropna()
resid_df

In [ ]:
X_test = sm.add_constant(resid_df['resid_diff_lag'])
ols_test = sm.OLS(resid_df['resid_diff'], X_test).fit(cov_type='HC1')
print("\nWooldridge test for serial correlation in panel data:")
print(f"  Coefficient on lagged diff: {ols_test.params.iloc[1]:.4f}")
print(f"  t-statistic: {ols_test.tvalues.iloc[1]:.4f}")
print(f"  p-value: {ols_test.pvalues.iloc[1]:.4f}")

# wald test

In [ ]:
pre_treatment_terms = [t for t in event_study_terms if int(t.split('_')[2]) < -1]
pre_coefs = event_study_model_reduced.params[pre_treatment_terms].values
pre_vcov = event_study_model_reduced.cov.loc[pre_treatment_terms, pre_treatment_terms].values

In [ ]:
q = len(pre_treatment_terms)
wald_stat = pre_coefs @ np.linalg.inv(pre_vcov) @ pre_coefs / q
df_resid = event_study_model_reduced.nobs - event_study_model_reduced.params.shape[0]
p_value = 1 - scipy_stats.f.cdf(wald_stat, q, df_resid)

print(f"\nJoint F-test for pre-treatment coefficients:")
print(f"  F({q}, {df_resid}) = {wald_stat:.4f}")
print(f"  p-value = {p_value:.4f}")

In [ ]:
(
    prepsm_data
    .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
    .filter(~pl.col('idind').is_in(alike_matched.implode()))
    .filter(pl.col('period_relevance') > -5)
).write_parquet('../data/panel_common_pre_psm_mean3_months_with_add_health_variables_no_outliers.parquet')

In [ ]:
post_psm = pl.read_parquet('../data/panel_common_post_psm_full_mean3_months_with_add_health_variables.parquet').filter(pl.col('is_employed_has_disablity') == 0)#.to_pandas().set_index(['idind', 'year'])
post_psm

In [ ]:
possible_spillovers = post_psm['idind']

In [ ]:
X_agg = post_psm[['is_employed_has_disablity']]
y = post_psm['net_job']

In [ ]:
agg_did_model = PanelOLS(y, X_agg, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)
print(agg_did_model.summary)

In [ ]:
(
    post_psm
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        #.filter(pl.col('working_pop') == 1)
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'is_employed_has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='is_employed_has_disablity')

In [ ]:
psm_matched_ids = pl.read_parquet('../data/panel_common_post_psm.parquet')['idind'].unique()
psm_matched_ids.implode()

In [ ]:
(
    panel_common_2019_2020
        .filter(pl.col('idind').is_in(psm_matched_ids.implode()))
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        .filter(pl.col('working_pop') == 1)
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/LOST-STATS/LOST-STATS.github.io/master/Model_Estimation/Data/Event_Study_DiD/bacon_example.csv")
df

In [ ]:
df['time_to_treat'] = (
    df['year'].sub(df['_nfd'])
        # missing values for _nfd implies no treatment
        .fillna(0)
        # so we don't have decimals in our factor names
        .astype('int')
)
df

# psm variables

In [ ]:
dictionary = pd.read_excel('../data/variable_dictionary.xlsx')
dictionary

In [ ]:
labels_df = dictionary[['question', 'label']].drop_duplicates()
label_dict = {}

for col, label in zip(labels_df['question'], labels_df['label']):
    label_dict[col] = label
    print(f'{col} : {label}')

In [ ]:
for col, label in zip(labels_df['question'], labels_df['label']):
    if 'disab' in label.lower():
        print(f'{col} : {label}')

In [ ]:
for col, label in zip(labels_df['question'], labels_df['label']):
    if 'm' in col.lower():
        print(f'{col} : {label}')